# 02 — Exploratory Data Analysis

**STAT 418 Final Project — NBA Playoff Underperformance Predictor**

This notebook explores the processed playoff dataset before model training. It is intended to be re-run end-to-end on a freshly built `data/processed/playoff_features.parquet`.

## Sections
1. Data shape & basic sanity checks
2. Game Score distributions across eras (motivates z-score normalization)
3. Target balance (underperform yes/no)
4. Feature-vs-target visualizations
5. Player-tier breakdowns (star vs role player)
6. Series-game-number patterns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

from src.data.compute_game_score import add_game_score_column

DATA = '../data/processed'

## 1. Load and sanity-check the modeling table

In [ ]:
df = pd.read_parquet(f'{DATA}/playoff_features.parquet')
print(f'Shape: {df.shape}')
print(f'Seasons: {df.season.min()} → {df.season.max()}')
print(f'Unique players: {df.player_id.nunique()}')
print(f'Underperform rate: {df.y_underperform.mean():.3f}')
df.head()

## 2. Game Score distributions across eras

This visualization motivates per-season z-score normalization. League-average production has risen sharply across the 22-season window.

In [ ]:
logs = pd.read_parquet(f'{DATA}/player_game_logs.parquet')
rs = logs[logs['season_type'] == 'Regular Season']
by_season = rs.groupby('season').agg(
    pts_avg=('pts', 'mean'),
    gs_avg=('game_score', 'mean'),
    fg3a_avg=('fg3a', 'mean'),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(axes, ['pts_avg', 'gs_avg', 'fg3a_avg'], 
                          ['League avg PTS/game', 'League avg Game Score/game', 'League avg 3PA/game']):
    ax.plot(by_season['season'], by_season[col], marker='o', color='#1E3A8A')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=70)
plt.tight_layout()

## 3. Target balance

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df.groupby('season')['y_underperform'].mean().plot(kind='bar', ax=ax, color='#DC2626')
ax.set_title('Playoff underperform rate per season (threshold = 1.0 GS)')
ax.set_ylabel('Fraction of player-games')
ax.tick_params(axis='x', rotation=70)

## 4. Feature-vs-target visualizations

In [ ]:
key_features = ['opp_def_rating_z', 'rs_game_score_std_z', 'series_game_number',
                'past_game7_gs_avg', 'gs_in_series_so_far_mean']
fig, axes = plt.subplots(1, len(key_features), figsize=(4 * len(key_features), 4))
for ax, col in zip(axes, key_features):
    if col not in df.columns:
        ax.set_visible(False)
        continue
    sns.boxplot(x='y_underperform', y=col, data=df, ax=ax,
                palette=['#2A9D8F', '#E76F51'])
    ax.set_title(col)
plt.tight_layout()

## 5. Player-tier breakdowns

Star players (top quartile of regular-season Game Score) vs role players.

In [ ]:
q75 = df['rs_game_score_mean'].quantile(0.75)
df['tier'] = np.where(df['rs_game_score_mean'] >= q75, 'Star', 'Role')
tier_counts = df.groupby(['tier', 'y_underperform']).size().unstack(fill_value=0)
tier_counts['underperform_rate'] = tier_counts[1] / (tier_counts[0] + tier_counts[1])
tier_counts

## 6. Series-game-number patterns

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
by_game = df.groupby('series_game_number')['y_underperform'].mean()
by_game.plot(kind='bar', ax=ax, color='#1E3A8A')
ax.set_title('Underperform rate by series_game_number')
ax.set_xlabel('Game # in series')
ax.set_ylabel('P(underperform)')

## Takeaways for modeling

- League-wide production trends strongly over 2003–2025 → per-season z-score normalization is justified.
- Target is moderately balanced (~40–50%).
- `opp_def_rating_z` and series-game-number show clear visual separation by target → these should be predictive.
- Stars and role players have noticeably different underperform rates → tier-conditional evaluation is necessary in `03_model_training.ipynb`.